# Глава 10: Внимание — это всё, что нужно

> "Внимание — это всё, что нужно." — **Vaswani и др.**, Google Research, 2017

---

## Что вы изучите

- Почему статических эмбеддингов недостаточно (слово "bank" в "river bank" и "savings bank")
- Что такое внимание на концептуальном уровне (токены смотрят друг на друга)
- Как построить самовнимание шаг за шагом на основе концепций Query, Key, Value
- Зачем масштабировать оценки внимания и маскировать будущие токены
- Как эффективно перейти от одноголового внимания к многоголовому
- Как объединить внимание с полносвязными сетями в полные блоки Transformer
- Как визуализировать возникающие паттерны внимания

---

## Настройка

Сначала установим необходимые пакеты:

In [ ]:
# Установка необходимых пакетов
!pip install -q torch transformers matplotlib

In [ ]:
# ===== ИМПОРТЫ =====
import torch                     # PyTorch: операции с тензорами
import torch.nn as nn            # Слои нейронной сети
import torch.nn.functional as F  # Математические функции (softmax, gelu)
import math                      # Для sqrt при масштабировании внимания
import matplotlib.pyplot as plt  # Визуализация
import numpy as np               # Операции с массивами

# Основные строительные блоки, которые мы будем использовать:
# - nn.Linear(in, out): Слой матричного умножения (обучает веса)
# - F.softmax(x, dim): Преобразует оценки в вероятности (сумма равна 1)
# - Оператор @: Матричное умножение (то же, что torch.matmul)

## 1. Почему статических эмбеддингов недостаточно

Проблема: эмбеддинги из главы 9 являются **статическими** — каждый токен получает один и тот же вектор независимо от контекста.

In [ ]:
# Имитация выхода из GPT2Embeddings главы 9
batch_size = 2
seq_len = 6
embed_dim = 768

# Эмбеддинги из главы 9 (случайные для этого примера, но представьте, что они настоящие)
embeddings = torch.randn(batch_size, seq_len, embed_dim)
print(f"Форма входных эмбеддингов: {embeddings.shape}")
# Ожидаемый вывод: torch.Size([2, 6, 768])

# Это СТАТИЧЕСКИЕ эмбеддинги из главы 9
# Наша цель: Преобразовать их в КОНТЕКСТНО-ЗАВИСИМЫЕ эмбеддинги

print("\nСлово 'bank' всегда получает один и тот же вектор:")
print("- 'river bank' → тот же эмбеддинг")
print("- 'savings bank' → тот же эмбеддинг")
print("- Но они означают совершенно разные вещи!")

## 2. Построение самовнимания шаг за шагом

Давайте построим внимание постепенно, показывая формы на каждом шаге.

**Интуиция:** Каждый токен спрашивает: "Какие другие токены важны для моего понимания?"
- **Запрос (Query, Q)**: "Что я ищу?"
- **Ключ (Key, K)**: "Что я предлагаю?"
- **Значение (Value, V)**: "Моя информация для обмена"

Представьте это как поисковую систему: запрос — это ваш поиск, ключи — это заголовки/теги документов, значения — это фактическое содержимое.

### Шаг 1: Создание проекций Query, Key, Value

**Что такое `nn.Linear(in_dim, out_dim)`?**
- Создаёт матрицу весов формы (in_dim, out_dim)
- Когда вы пропускаете через неё вход: `output = input @ weight`
- Эти веса "обучаемые" — они обновляются в процессе обучения

In [ ]:
# Используем меньшую размерность для ясности
d_model = 768  # Из эмбеддингов (глава 9)
d_k = 64       # Размерность для Q, K, V (типично: d_model / num_heads)

# Создаём слои проекций (у них есть обучаемые параметры!)
W_q = nn.Linear(d_model, d_k)  # Проекция запроса (смещение сохранено, как в GPT-2)
W_k = nn.Linear(d_model, d_k)  # Проекция ключа
W_v = nn.Linear(d_model, d_k)  # Проекция значения

# Проецируем эмбеддинги в Q, K, V
# Почему Linear? Она обучает лучшее преобразование для каждой роли
Q = W_q(embeddings)  # (batch, seq, d_k) = (2, 6, 64)
K = W_k(embeddings)  # (batch, seq, d_k) = (2, 6, 64)
V = W_v(embeddings)  # (batch, seq, d_k) = (2, 6, 64)

print(f"Форма Q: {Q.shape}")  # Ожидается: torch.Size([2, 6, 64])
print(f"Форма K: {K.shape}")  # Ожидается: torch.Size([2, 6, 64])
print(f"Форма V: {V.shape}")  # Ожидается: torch.Size([2, 6, 64])

print("\nТеперь каждый токен имеет:")
print("- Вектор Q (64 измерения): 'Что я ищу'")
print("- Вектор K (64 измерения): 'Что я предлагаю'")
print("- Вектор V (64 измерения): 'Моя информация для обмена'")

### Шаг 2: Вычисление оценок внимания (Q · K^T)

In [ ]:
# Вычисляем оценки внимания: Q @ K^T
# Нужно транспонировать K, чтобы размерности совпали для матричного умножения

# Форма Q: (batch, seq, d_k) = (2, 6, 64)
# Форма K: (batch, seq, d_k) = (2, 6, 64)

# K.transpose(-2, -1) меняет местами последние два измерения:
# Отрицательные индексы: -1 = последнее измерение, -2 = предпоследнее
# Таким образом K переходит из (2, 6, 64) → (2, 64, 6)

# Матричное умножение: (2, 6, 64) @ (2, 64, 6) → (2, 6, 6)
scores = Q @ K.transpose(-2, -1)

print(f"Форма Q: {Q.shape}")
print(f"Форма K: {K.shape}")
print(f"Форма транспонированного K: {K.transpose(-2, -1).shape}")
print(f"Форма оценок внимания: {scores.shape}")
# Ожидается: torch.Size([2, 6, 6])

print(f"\nОценки для первого элемента батча:")
print(scores[0])
print("\nМатрица 6×6, где элемент [i,j] = насколько токен i обращает внимание на токен j")

### Шаг 3: Масштабирование оценок

In [ ]:
# Масштабируем на sqrt(размерность)
# Почему sqrt? Математическое доказательство показывает, что это сохраняет стабильную дисперсию
scores = scores / math.sqrt(d_k)

print(f"Форма масштабированных оценок: {scores.shape}")  # Всё ещё (2, 6, 6)
print(f"Теория: при единичной дисперсии элементов Q и K дисперсия оценок примерно {d_k}")
print(f"После масштабирования на sqrt({d_k}) = {math.sqrt(d_k):.2f} теория говорит: дисперсия примерно 1")

print("\nПочему это важно: Без масштабирования высокоразмерное внимание")
print("поместило бы почти весь вес на один токен, теряя преимущество")
print("обращения внимания на несколько токенов.")

### Шаг 4: Применение Softmax для получения весов внимания

Это теоретические значения, а не измерения нашей демонстрации: математика предполагает единичную дисперсию элементов Q и K. Наши реальные оценки получаются из `nn.Linear`-проекций случайных эмбеддингов, их измеренная дисперсия намного ниже 64. Важно отношение: масштабирование уменьшает дисперсию в `d_k` раз независимо от исходного значения.


In [ ]:
# Применяем softmax по последнему измерению (по ключам)
# Это делает каждую строку (каждый запрос) в сумме равной 1
attn_weights = F.softmax(scores, dim=-1)

print(f"Форма весов внимания: {attn_weights.shape}")  # Ожидается: (2, 6, 6)
print(f"\nВеса внимания для токена 0 (первый батч):")
print(attn_weights[0, 0])
# Пример вывода: tensor([0.15, 0.20, 0.30, 0.18, 0.10, 0.07])
# Их сумма равна 1.0!

print(f"\nСумма весов для токена 0: {attn_weights[0, 0].sum().item():.4f}")
# Ожидается: Сумма весов для токена 0: 1.0000

### Шаг 5: Взвешенная сумма значений

In [ ]:
# Веса внимания: (batch, seq, seq) = (2, 6, 6)
# Значения:      (batch, seq, d_k)  = (2, 6, 64)
# Хотим:         (batch, seq, d_k)  = (2, 6, 64)

output = attn_weights @ V

print(f"Форма выхода: {output.shape}")  # Ожидается: torch.Size([2, 6, 64])

print(f"\nИсходный эмбеддинг для токена 0 (первые 10 измерений):")
print(embeddings[0, 0, :10])

print(f"\nВыход после внимания для токена 0 (первые 10 измерений):")
print(output[0, 0, :10])
print("\nРазные значения! Этот токен включил контекст")

### Полная функция внимания

Давайте упакуем 5 шагов в одну функцию:

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Вычислить масштабированное скалярное произведение внимания.
    
    Args:
        Q: Запросы (batch, seq, d_k)
        K: Ключи   (batch, seq, d_k)
        V: Значения (batch, seq, d_k)
        mask: Опциональная маска (batch, seq, seq)
    
    Returns:
        output: Выход внимания (batch, seq, d_k)
        attn_weights: Веса внимания (batch, seq, seq)
    """
    d_k = Q.size(-1)  # Получить размерность запросов/ключей
    
    # Шаг 1: Вычислить оценки Q @ K^T
    scores = Q @ K.transpose(-2, -1)  # (batch, seq, seq)
    
    # Шаг 2: Масштабировать на sqrt(d_k)
    scores = scores / math.sqrt(d_k)
    
    # Шаг 3: Применить маску, если она предоставлена
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Шаг 4: Softmax для получения весов внимания
    attn_weights = F.softmax(scores, dim=-1)  # (batch, seq, seq)
    
    # Шаг 5: Взвешенная сумма значений
    output = attn_weights @ V  # (batch, seq, d_k)
    
    return output, attn_weights


# Тестируем
output, attn_weights = scaled_dot_product_attention(Q, K, V)

print(f"Форма выхода: {output.shape}")  # Ожидается: (2, 6, 64)
print(f"Форма весов внимания: {attn_weights.shape}")  # Ожидается: (2, 6, 6)
print(f"\nРаспределение внимания первого токена:")
print(attn_weights[0, 0])

## 3. Причинное маскирование для авторегрессионной генерации

### Проблема читерства

Без маскирования при обработке "cat" модель может обращать внимание на ВСЕ токены, включая будущие! Это читерство во время обучения.

In [ ]:
def create_causal_mask(seq_len):
    """
    Создать причинную маску: верхний треугольник False (блокировать), нижний True (разрешить).
    
    Returns:
        mask: (seq_len, seq_len) булев тензор
    """
    # torch.tril создаёт нижнетреугольную матрицу
    # 1 ниже диагонали (включая диагональ), 0 выше
    mask = torch.tril(torch.ones(seq_len, seq_len))
    
    return mask


# Пример с seq_len = 6
mask = create_causal_mask(6)
print("Причинная маска (1 = разрешить, 0 = блокировать):")
print(mask)

print("\nЧтение маски:")
print("- Строка 0 (токен 0): Может обращать внимание только на столбец 0")
print("- Строка 2 (токен 2): Может обращать внимание на столбцы 0, 1, 2")
print("- Строка 5 (токен 5): Может обращать внимание на все столбцы 0-5")

### Применение маски

In [ ]:
# Тестируем с причинной маской
Q_test = torch.randn(2, 6, 64)  # (batch, seq, d_k)
K_test = torch.randn(2, 6, 64)
V_test = torch.randn(2, 6, 64)

# Создаём причинную маску
causal_mask = create_causal_mask(6)  # (seq, seq)

# Применяем внимание с маской
output_masked, attn_weights_masked = scaled_dot_product_attention(
    Q_test, K_test, V_test, mask=causal_mask
)

print("Веса внимания С причинной маской (первый элемент батча):")
print(attn_weights_masked[0])

print("\nОбратите внимание: Верхний треугольник полностью нулевой! Нет внимания к будущему!")

## 4. Многоголовое внимание

### Зачем несколько голов?

Одна голова внимания может захватить только ОДИН тип отношений одновременно. Несколько голов позволяют модели изучать разные паттерны одновременно:

| Голова | Что может изучить |
|--------|-------------------|
| Голова 1 | Отношения субъект-глагол ("cat" → "sat") |
| Голова 2 | Связи прилагательное-существительное ("lazy" → "dog") |
| Голова 3 | Паттерны соседних слов (локальный контекст) |
| Голова 4 | Дальние зависимости (разрешение местоимений) |

**Ключевая идея:** 12 голов по 64 измерения каждая = 768 общих измерений = то же, что 1 большая голова!
Никаких дополнительных параметров — просто разные перспективы.

### Эффективная реализация

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Эффективное многоголовое внимание (все головы обрабатываются вместе).
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model должен делиться на num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads  # 768 / 12 = 64
        
        # Объединённая QKV проекция (в 3 раза эффективнее отдельных!)
        # Почему 3 * d_model? Потому что мы проецируем в Q, K, V одновременно
        # bias=True соответствует GPT-2, поэтому позже мы сможем загрузить его предобученные смещения
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)

        # Выходная проекция
        self.out_proj = nn.Linear(d_model, d_model)
        
        # Dropout на весах внимания
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape
        
        # ===== Шаг 1: Проекция в Q, K, V (всё сразу!) =====
        qkv = self.qkv_proj(x)  # (batch, seq, 3 * d_model)
        
        # ===== Шаг 2: Разделение на Q, K, V и изменение формы для многоголового =====
        # Изменяем форму на (batch, seq, 3, num_heads, d_head)
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)
        
        # Переставляем в (3, batch, num_heads, seq, d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        
        # Разделяем на Q, K, V: каждый имеет форму (batch, num_heads, seq, d_head)
        Q, K, V = qkv[0], qkv[1], qkv[2]
        
        # ===== Шаг 3: Масштабированное скалярное произведение внимания (батчи по головам) =====
        d_k = self.d_head
        scores = Q @ K.transpose(-2, -1)  # (batch, num_heads, seq, seq)
        scores = scores / math.sqrt(d_k)
        
        # Применяем причинную маску, если она предоставлена
        if mask is not None:
            # Расширяем маску для голов: (seq, seq) → (1, 1, seq, seq)
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)  # (batch, num_heads, seq, seq)
        attn_weights = self.dropout(attn_weights)
        
        # Взвешенная сумма значений
        attn_output = attn_weights @ V  # (batch, num_heads, seq, d_head)
        
        # ===== Шаг 4: Конкатенация голов =====
        # Транспонируем в (batch, seq, num_heads, d_head)
        attn_output = attn_output.transpose(1, 2)
        
        # Изменяем форму на (batch, seq, d_model), это конкатенирует головы
        attn_output = attn_output.reshape(batch, seq, d_model)
        
        # ===== Шаг 5: Финальная проекция =====
        output = self.out_proj(attn_output)
        
        return output, attn_weights


# Тестируем

## 5. Полные блоки Transformer

### Полносвязная сеть

Полносвязная сеть расширяет размерность (768 → 3072), применяет нелинейность, затем сжимает обратно (3072 → 768).

**Что такое GELU?**
- GELU (Gaussian Error Linear Unit) — это функция активации
- Похожа на ReLU, но более гладкая — не имеет жёсткого обрезания в нуле
- Используется в GPT-2, BERT и большинстве современных трансформеров
- Интуиция: "пропускает" сигнал в зависимости от величины входа

In [ ]:
class FeedForward(nn.Module):
    """
    Позиционная полносвязная сеть.
    Применяется к каждой позиции независимо (одинаковые веса для всех позиций).
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        """
        Args:
            d_model: Размерность модели (768 для GPT-2 small)
            d_ff: Размерность полносвязной сети (обычно 4 * d_model = 3072)
            dropout: Вероятность dropout
        """
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x: (batch, seq, d_model)
        x = self.fc1(x)        # (batch, seq, d_ff), расширение
        x = F.gelu(x, approximate="tanh")  # Нелинейность (вариант GELU из GPT-2)
        x = self.dropout(x)
        x = self.fc2(x)        # (batch, seq, d_model), проекция обратно
        return x


# Тестируем
ffn = FeedForward(d_model=768, d_ff=3072, dropout=0.1)
x_test = torch.randn(2, 6, 768)
output_ffn = ffn(x_test)

print(f"Форма входа:  {x_test.shape}")      # Ожидается: (2, 6, 768)
print(f"Форма выхода: {output_ffn.shape}")  # Ожидается: (2, 6, 768)

### Полный блок Transformer (стиль пре-нормализации)

**Пре-нормализация против пост-нормализации:**
- **Пост-нормализация** (оригинальный Transformer): LayerNorm ПОСЛЕ каждого подслоя
- **Пре-нормализация** (GPT-2, современные): LayerNorm ПЕРЕД каждым подслоем

Почему пре-нормализация? Она делает обучение более стабильным для глубоких сетей (12+ слоёв). Градиенты текут более плавно через остаточные соединения.

**Остаточные соединения:** `output = x + sublayer(x)`
- Создают "автострады" для обратного потока градиентов
- Без остаточных соединений градиенты исчезают в глубоких сетях

In [ ]:
class TransformerBlock(nn.Module):
    """
    Полный блок Transformer с многоголовым вниманием, полносвязной сетью,
    остаточными соединениями и нормализацией слоя (стиль пре-нормализации как в GPT-2).
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        """
        Args:
            d_model: Размерность модели (768)
            num_heads: Количество голов внимания (12)
            d_ff: Размерность полносвязной сети (3072 = 4 * d_model)
            dropout: Вероятность dropout
        """
        super().__init__()
        
        # Нормализация слоя (перед каждым подслоем)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        
        # Многоголовое внимание
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Полносвязная сеть
        self.ffn = FeedForward(d_model, d_ff, dropout)
        
        # Dropout (применяется после каждого подслоя)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        """
        Args:
            x: Входные эмбеддинги (batch, seq, d_model)
            mask: Причинная маска (seq, seq) или (1, 1, seq, seq)
        
        Returns:
            x: Выходные эмбеддинги (batch, seq, d_model)
            attn_weights: Веса внимания (batch, heads, seq, seq)
        """
        # ===== Многоголовое внимание с остаточным соединением =====
        # Пре-нормализация: Нормализуем ПЕРЕД вниманием
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)  # Остаточное соединение
        
        # ===== Полносвязная сеть с остаточным соединением =====
        # Пре-нормализация: Нормализуем ПЕРЕД полносвязной сетью
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)  # Остаточное соединение
        
        return x, attn_weights


# Тестируем полный блок Transformer

## 6. Визуализация паттернов внимания

In [ ]:
def visualize_attention(attn_weights, tokens, head_idx=0, ax=None):
    """
    Визуализировать веса внимания как тепловую карту для конкретной головы.
    
    Args:
        attn_weights: Веса внимания (batch, heads, seq, seq)
        tokens: Список строк токенов (длина = seq)
        head_idx: Какую голову визуализировать
        ax: Ось Matplotlib (если None, создать новую фигуру)
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    # Извлекаем веса для указанной головы (первый элемент батча)
    weights = attn_weights[0, head_idx].detach().cpu().numpy()
    
    # Строим тепловую карту
    im = ax.imshow(weights, cmap='viridis', aspect='auto', vmin=0, vmax=1)
    
    # Устанавливаем метки и подписи
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right')
    ax.set_yticklabels(tokens)
    
    # Подписи
    ax.set_xlabel('Ключ (на что обращают внимание)', fontsize=10)
    ax.set_ylabel('Запрос (что обращает внимание)', fontsize=10)
    ax.set_title(f'Веса внимания — Голова {head_idx}', fontsize=12)
    
    # Цветовая шкала
    plt.colorbar(im, ax=ax, label='Вес внимания')
    
    return ax


# Пример: Обрабатываем настоящее предложение
from transformers import AutoTokenizer

# Токенизация
tokenizer = AutoTokenizer.from_pretrained("gpt2")
text = "The quick brown fox jumps over the lazy dog"
token_ids = tokenizer.encode(text, return_tensors="pt")  # (1, seq)

# Получаем строки токенов
tokens = [tokenizer.decode([t]) for t in token_ids[0]]
print(f"Токены: {tokens}")

# Пропускаем через слой эмбеддинга (имитируем выход главы 9)
embed_layer = nn.Embedding(50257, 768)
embeddings_viz = embed_layer(token_ids)  # (1, seq, 768)

# Создаём причинную маску
seq_len_viz = embeddings_viz.size(1)
mask_viz = create_causal_mask(seq_len_viz)

# Пропускаем через блок Transformer
block_viz = TransformerBlock(d_model=768, num_heads=12, d_ff=3072, dropout=0.1)
output_viz, attn_weights_viz = block_viz(embeddings_viz, mask_viz)

print(f"\nФорма весов внимания: {attn_weights_viz.shape}")  # Ожидается: (1, 12, seq, seq)

# Визуализируем первые 4 головы
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for i, ax in enumerate(axes.flat):
    visualize_attention(attn_weights_viz, tokens, head_idx=i, ax=ax)

plt.tight_layout()
plt.show()

print("\nЧто вы видите:")
print("1. Нижнетреугольный паттерн (причинное маскирование работает!)")
print("2. Разные паттерны для каждой головы (разнообразие — это хорошо)")
print("3. Некоторые головы фокусируются локально, другие на дальних зависимостях")

## 7. Полный конвейер

Давайте проследим данные от сырого текста до выхода блока Transformer:

In [ ]:
# GPT2Embeddings из главы 9 для автономного запуска этого блокнота
if 'GPT2Embeddings' not in globals():
    class GPT2Embeddings(nn.Module):
        """Эмбеддинги токенов + позиций (класс из главы 9)."""
        def __init__(self, vocab_size, max_seq_len, embed_dim):
            super().__init__()
            self.token_embed = nn.Embedding(vocab_size, embed_dim)
            self.pos_embed = nn.Embedding(max_seq_len, embed_dim)
            nn.init.normal_(self.token_embed.weight, mean=0.0, std=0.02)
            nn.init.normal_(self.pos_embed.weight, mean=0.0, std=0.02)
            self.max_seq_len = max_seq_len

        def forward(self, token_ids):
            batch_size, seq_len = token_ids.shape
            if seq_len > self.max_seq_len:
                raise ValueError(f"Sequence length {seq_len} exceeds max_seq_len {self.max_seq_len}")
            token_embeddings = self.token_embed(token_ids)
            position_ids = torch.arange(seq_len, device=token_ids.device)
            position_ids = position_ids.unsqueeze(0).expand(batch_size, seq_len)
            return token_embeddings + self.pos_embed(position_ids)

# ===== Глава 8: Токенизация =====
from transformers import AutoTokenizer
text = "The cat sat on the mat"  # Сырой текст
tokenizer = AutoTokenizer.from_pretrained("gpt2")
token_ids = tokenizer.encode(text, return_tensors="pt")  # (1, 6)

print("Шаг 1: Токенизация")
print(f"Текст: {text}")
print(f"ID токенов: {token_ids}")
print(f"Форма: {token_ids.shape}\n")

# ===== Глава 9: Эмбеддинги =====
# GPT2Embeddings из главы 9 (определён ниже, если вы пропустили тот блокнот)
embed_layer = GPT2Embeddings(vocab_size=50257, max_seq_len=1024, embed_dim=768)
embeddings = embed_layer(token_ids)  # (1, 6, 768)

print("Шаг 2: Эмбеддинг")
print(f"Форма эмбеддингов: {embeddings.shape}")
print(f"Эмбеддинг первого токена (первые 10 измерений): {embeddings[0, 0, :10]}\n")

# ===== Глава 10: Внимание (ЭТА ГЛАВА) =====
# Создаём причинную маску
seq_len_final = token_ids.size(1)  # 6
mask_final = create_causal_mask(seq_len_final)

# Блок Transformer
block_final = TransformerBlock(d_model=768, num_heads=12, d_ff=3072, dropout=0.1)
output_final, attn_weights_final = block_final(embeddings, mask_final)  # (1, 6, 768)

print("Шаг 3: Внимание (ЭТА ГЛАВА)")
print(f"Форма выхода: {output_final.shape}")
print(f"Первый токен после внимания (первые 10 измерений): {output_final[0, 0, :10]}\n")

print("Конвейер завершён!")
print("Сырой текст → Токены → Эмбеддинги → Внимание → Контекстно-зависимые векторы")

print("\n===== Предпросмотр главы 11: Складываем 12 блоков =====")
print("В главе 11 мы пропустим выход через ещё 11 блоков Transformer!")

## Упражнения

### Упражнение 1: Ручной расчёт внимания

Даны матрицы Q, K, V. Вручную вычислите оценки внимания, примените softmax и получите выход. Проверьте, что ваши вычисления совпадают с `scaled_dot_product_attention()`.

In [ ]:
# Создаём простые 2×3 Q, K, V (batch=1, seq=2, d_k=3)
Q_ex = torch.tensor([[[1.0, 0.0, 1.0], [0.0, 1.0, 1.0]]])  # (1, 2, 3)
K_ex = torch.tensor([[[1.0, 1.0, 0.0], [0.0, 1.0, 1.0]]])  # (1, 2, 3)
V_ex = torch.tensor([[[2.0, 0.0, 1.0], [1.0, 2.0, 0.0]]])  # (1, 2, 3)

# ВАШ КОД ЗДЕСЬ:
# 1. Вычислите scores = Q @ K^T
# 2. Масштабируйте на sqrt(d_k)
# 3. Примените softmax
# 4. Взвешенная сумма с V
# 5. Сравните с scaled_dot_product_attention()

### Упражнение 2: Проверка причинной маски

Создайте веса внимания с причинным маскированием и без него. Проверьте, что верхний треугольник нулевой при маскировании.

In [ ]:
# ВАШ КОД ЗДЕСЬ:
# 1. Создайте Q, K, V для seq_len=5
# 2. Вычислите внимание БЕЗ маски
# 3. Вычислите внимание С причинной маской
# 4. Напечатайте обе матрицы весов внимания
# 5. Проверьте, что верхний треугольник нулевой в замаскированной версии

### Упражнение 3: Формы многоголового внимания

Проследите преобразования форм через `MultiHeadAttention` с конкретными числами.

In [ ]:
# ВАШ КОД ЗДЕСЬ:
# Создайте MHA с d_model=512, num_heads=8, seq_len=10
# Напечатайте форму после каждого шага:
# 1. Вход
# 2. После QKV проекции
# 3. После изменения формы для разделения голов
# 4. После вычисления внимания
# 5. После конкатенации голов
# 6. После выходной проекции

### Упражнение 4: Сравнение одно- и многоголового внимания

Пропустите один и тот же вход через одноголовое и 12-головое внимание. Сравните количество параметров.

In [ ]:
# ВАШ КОД ЗДЕСЬ:
# 1. Создайте одноголовое внимание (d_model=768, num_heads=1)
# 2. Создайте многоголовое внимание (d_model=768, num_heads=12)
# 3. Подсчитайте параметры в каждом
# 4. Пропустите один и тот же вход через оба
# 5. Сравните выходы и количество параметров

### Упражнение 5: Визуализация внимания

Обработайте своё собственное предложение и визуализируйте различные головы внимания.

In [ ]:
# ВАШ КОД ЗДЕСЬ:
# 1. Выберите интересное предложение (например, "Alice gave Bob a book")
# 2. Токенизируйте его
# 3. Пропустите через TransformerBlock
# 4. Визуализируйте головы 0, 3, 7, 11
# 5. Опишите, какие паттерны вы видите (локальные против дальних)

### Упражнение 6: Складывание блоков

Сложите 3 блока Transformer и обработайте последовательность через все из них.

In [ ]:
# ВАШ КОД ЗДЕСЬ:
# 1. Создайте 3 отдельных экземпляра TransformerBlock
# 2. Пропустите эмбеддинги через блок1 → блок2 → блок3
# 3. Напечатайте форму после каждого блока
# 4. Сравните входные эмбеддинги с финальным выходом
# 5. Насколько они отличаются?

### Упражнение 7: Пре-нормализация против пост-нормализации

Реализуйте блок Transformer с пост-нормализацией и сравните с пре-нормализацией.

In [ ]:
# ВАШ КОД ЗДЕСЬ:
# 1. Реализуйте TransformerBlockPostNorm, где LayerNorm идёт ПОСЛЕ
# 2. Пропустите один и тот же вход через пре-нормализацию и пост-нормализацию
# 3. Сравните выходы
# 4. У какого более стабильные градиенты? (можете проверить величины градиентов)

### Упражнение 8: Подсчёт параметров

Вычислите точное количество параметров для одного блока Transformer.

In [ ]:
# ВАШ КОД ЗДЕСЬ:
# Для d_model=768, num_heads=12, d_ff=3072:
# 1. Подсчитайте параметры QKV проекции
# 2. Подсчитайте параметры выходной проекции
# 3. Подсчитайте параметры полносвязной сети (fc1 + fc2)
# 4. Подсчитайте параметры LayerNorm (2 слоя)
# 5. Суммируйте для получения общего количества
# 6. Проверьте с помощью model.parameters()

## Резюме главы

**Что мы построили:**

1. Масштабированное скалярное произведение внимания (Q, K, V → оценки → softmax → взвешенная сумма)
2. Причинное маскирование для авторегрессионной генерации (никакого подглядывания в будущее)
3. Эффективное многоголовое внимание (1 голова → 12 голов через трюки с изменением формы)
4. Полные блоки Transformer (внимание + полносвязная сеть + остаточные + нормализация слоя)
5. Визуализация внимания (тепловые карты, показывающие на что обращает внимание модель)

**Основные концепции:**

- **Статические эмбеддинги** (глава 9) → **Контекстно-зависимые представления** (глава 10)
- **Запрос/Ключ/Значение**: Три обученные проекции, выполняющие разные роли
- **Веса внимания**: Softmax-вероятности, показывающие релевантность
- **Многоголовое**: Разные головы изучают разные паттерны (никаких дополнительных параметров!)
- **Остаточные + Нормализация**: Позволяют строить глубокие сети (100+ слоёв)

**Далее:** Глава 11 сложит эти блоки и добавит голову языкового моделирования для создания полной модели GPT!